# [Optional] Video Ingestion → Delta `videos` Table

**Purpose:** Build a queryable Unity Catalog reference table of video file metadata. This notebook is optional — the main training track (notebooks 01–03) does not depend on it.

**Why Delta, not Lance?** The `videos` table is pure structured metadata: strings, numbers, timestamps. It has no binary data, no embeddings, and will never be accessed by a training DataLoader. Delta is the right default here — it gives you Unity Catalog namespace (`catalog.schema.videos`), lineage tracking, Photon-accelerated queries, and zero extra setup. Lance would add complexity with no benefit for this access pattern.

**What this notebook does:**

1. **Scan the Volumes directory.** List all video files under `/Volumes/{catalog}/{schema}/{volume}/videos/` using `dbutils.fs.ls()` or equivalent.

2. **Extract file-level metadata per video.** For each file, use `ffprobe` (via `ffmpeg-python`) or `decord` to read metadata without decoding pixel data: filename, duration, fps, resolution (width × height), codec, and file size. This is a CPU-only operation and does not require GPU workers.

3. **Write to a Delta table.** Collect metadata into a Spark DataFrame and write to a Delta table registered in Unity Catalog. Use `mode='append'` to support incremental ingestion as new videos are added to the Volume.

4. **Verify.** Query the table via `spark.table('catalog.schema.videos')` to confirm row counts and schema. Show a sample of rows.

**Inputs:** Raw video files in `/Volumes/{catalog}/{schema}/{volume}/videos/`

**Outputs:** Delta table `{catalog}.{schema}.videos` in Unity Catalog.

**Use when:** You need SQL-queryable video-level metadata — e.g., filtering by duration or resolution before deciding which videos to extract frames from, or joining video metadata with downstream training results.